# gimVI: Spatial Gene Imputation (Pseudo-6K Benchmark)

Uses gimVI (scVI-based generative model) to jointly model spatial and scRNA-seq data,
imputing the missing ~12K genes in the pseudo-6K CosMx experiment.

**Input**: WTX CosMx h5ad; 6K gene list  
**Output**: `half_ori_half_gimvi_imp_log1p.csv` — combined 6K observed + imputed 12K genes


In [ ]:
import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scvi.external import GIMVI

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = "/path/to/cosmx/data"
OUT_DIR  = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX CosMx data (log1p normalized)
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad")
adata.obs['labels']  = adata.obs['aucell_cell_category_from_type'].tolist()
adata.obs_names      = adata.obs['fov_cell_id'].tolist()
adata.obs['fov']     = adata.obs['fov'].astype(int)

# Train/test split by FOV
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

adata_train = adata[train_cells, :].copy()  # full transcriptome (spatial reference)
adata_test  = adata[test_cells, :].copy()
adata_test.obs['batch'] = 0

In [ ]:
# 6K gene panel (pseudo-6K: restrict test data to 6K genes)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()
test_data_partial = adata_test[:, adata_test.var.index.isin(genes_6k)].copy()
print(f"Test genes (6K panel): {test_data_partial.n_vars:,}  |  Imputation targets: {adata_train.n_vars - test_data_partial.n_vars:,}")

In [ ]:
# Set up gimVI: spatial data (6K) + sequencing reference (full transcriptome)
GIMVI.setup_anndata(test_data_partial, labels_key="labels", batch_key="batch")
GIMVI.setup_anndata(adata_train,       labels_key="labels")

model = GIMVI(adata_train, test_data_partial)
model.train(500, datasplitter_kwargs={"drop_last": True})

In [ ]:
# Impute full transcriptome for test cells
# imputed[0] = train predictions, imputed[1] = test predictions
imputed = model.get_imputed_values(normalized=False)
pred_res = imputed[1]

adata_pred = sc.AnnData(X=pred_res, var=pd.DataFrame(index=adata.var_names.tolist()), obs=adata_test.obs)
adata_pred.obsm = adata_test.obsm
adata_pred.uns  = adata_test.uns
adata_pred.obsp = adata_test.obsp

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────

# Full predictions (all genes, imputed by gimVI)
gimvi_pred = pd.DataFrame(adata_pred.X)
gimvi_pred.index   = adata_pred.obs['fov_cell_id'].tolist()
gimvi_pred.columns = adata_pred.var.index.tolist()

# Observed 6K expression (raw log1p counts)
raw_6k = pd.DataFrame(adata_test[:, adata_test.var.index.isin(genes_6k)].X)
raw_6k.columns = [g for g in adata_test.var_names if g in genes_6k]
raw_6k.index   = adata_test.obs['fov_cell_id'].tolist()

# Combine: keep 6K observed, use gimVI for remaining 12K
imputed_genes = [g for g in gimvi_pred.columns if g not in genes_6k]
combined = pd.concat([raw_6k, gimvi_pred[imputed_genes]], axis=1)
combined.to_csv(f"{OUT_DIR}/half_ori_half_gimvi_imp_log1p.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")